# 06. Walk-forward forecasts, development choices, and the experiment freeze

**The question.** Using development data only, what static duration and updating half-life should the benchmarks use, and how do the forecast models behave? Then: freeze every choice before any test-period result is computed.

**Why a portfolio manager cares.** A manager trusts a backtest only if the rules were fixed before the test data were seen. This notebook is that commitment, with hashes that make later changes visible.

**Learning objectives**

1. Compute the static duration from development data.
2. Choose the EWMA half-life by the declared objective on development backtests.
3. Run every model walk-forward on development data and compare forecast errors.
4. Write the freeze record (configuration, code, and choices, all hashed).
5. Compute test-period forecasts with the frozen settings, stamped with their information times.
6. Run the look-ahead perturbation check on the real data.

| | |
|---|---|
| Inputs | `data/processed/daily_panel.parquet`, `data/processed/monthly_panel.parquet` |
| Outputs | `results/dev/*`, `results/freeze/freeze_record.json`, `results/forecasts/forecasts_{dev,test}.csv`, `results/forecasts/lookahead_check.json` |
| Run after | 05 |
| Approximate runtime | about 1 minute |

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import pipeline as pl
from mbshedge import models as md
from mbshedge import evaluation as ev
from mbshedge import freeze as fz
from mbshedge.sources import RawStore, load_raw_tables
daily = pd.read_parquet(paths.processed / "daily_panel.parquet")
monthly = pd.read_parquet(paths.processed / "monthly_panel.parquet")
win = pl.windows(daily, EXP)
ddir = paths.results / "dev"; ddir.mkdir(parents=True, exist_ok=True)
fdir = paths.results / "forecasts"; fdir.mkdir(parents=True, exist_ok=True)
print("development decisions:", win["dev"]["first_decision"], "to", win["dev"]["last_decision"],
      "| test decisions:", win["test"]["first_decision"], "to", win["test"]["last_decision"])
data_end = daily.index.max()
print(f"The data end on {data_end.date()}. The test period ends with {win['test_end']}, the last month whose final trading day is in the data.")
n_after = int((daily.index > win["test"]["end_date"]).sum())
if n_after:
    print(f"The {n_after} trading days after {win['test']['end_date'].date()} are not evaluated, because their month is not complete "
          "in the data. FRED usually posts Treasury yields one business day late, so a month becomes complete about a day after it ends.")

## 1. The static hedge

Static duration = the average of the valid realized monthly durations from April 2007 to December 2014. Each month counts once, so crisis months with large yield moves do not dominate. The static hedge then keeps this duration target for the whole test period (its notional changes a little each month only because the hedge bond's own duration changes with the yield level).

In [ ]:
static_d = md.static_duration(monthly, EXP["sample"]["first_month"], EXP["sample"]["development_end"])
print(f"Static duration (development average): {static_d:.3f} years")

## 2. Choosing the updating half-life (development backtests only)

For each candidate half-life we run the updating hedge through the common backtest on development decisions (April 2010 to December 2014) at the base settings, and score it with the declared objective

$$
J = \text{IR}_{\text{hurdle}} \times \sigma_{\text{gross}} + \text{annual cost drag},
$$

the same trade-off that defines "survives costs" later. Short half-lives react fast but trade more. Long half-lives trade less but lag.

In [ ]:
halflife, hl_table = pl.select_halflife(daily, monthly, win, EXP, static_d)
hl_table.to_csv(ddir / "halflife_selection.csv", index=False)
print(f"Selected half-life: {halflife} trading days")
hl_table.round(3)

## 3. Development-period forecasts

Every model forecasts next month's realized duration at each development month-end, refit on an expanding window. Forecasts of the economic and hybrid models before mid-2013 use the **retrospective** proxy incentive, so their development scores are labeled and serve only as a description.

In [ ]:
models_list = [m for m in EXP["models"]["list"]]
fc_dev = pl.forecasts(monthly, win, "dev", static_d, halflife, models_list)
fc_dev.to_csv(fdir / "forecasts_dev.csv", index=False)
tab_dev = ev.forecast_table(fc_dev, [m for m in models_list if m != "X"], reference="U")
tab_dev.to_csv(ddir / "forecast_table_development.csv", index=False)
print(f"Development forecast comparison on {tab_dev['n_months'].iloc[0]} common months "
      f"(share of decisions on the retrospective proxy: {(fc_dev.drop_duplicates('decision_month')['incentive_source'] == 'proxy_retrospective').mean():.0%})")
tab_dev.round(3)

## 4. The freeze

Everything chosen so far, plus SHA-256 hashes of the frozen configuration block, of the result-relevant source code (including the outcome mapping in `evaluation.classify_outcome`), and of the code cells of the analysis notebooks (02 and 05 to 09), goes into `results/freeze/freeze_record.json`. If the record already exists (a rerun), this cell verifies that the rerun reproduces every choice exactly and that nothing hashed has changed. Before any test-period computation has been logged, a record that no longer matches is replaced and the replacement is logged. After that, any change stops the run unless it is recorded in `docs/post_freeze_changes.md` and the run is repeated with `MBSHEDGE_ALLOW_POST_FREEZE=1`.

In [ ]:
info = json.loads((paths.results / "state" / "state_info.json").read_text())
dev_choices = {
    "static_duration_years": float(static_d),
    "ewma_halflife_days": int(halflife),
    "halflife_selection": hl_table.round(10).to_dict(orient="records"),
    "proxy_halflife_years": info["proxy"].get("halflife_years"),
    "proxy_offset_pp": info["proxy"].get("offset"),
    "sigma_pp": info["sigma_pp"],
    "theta_pp": info["theta_pp"],
    "nmdb_markets": info["markets"],
    "nmdb_lag_quarters": info["lag_quarters"],
    "first_dev_decision": str(win["dev"]["first_decision"]),
    "first_test_decision": str(win["test"]["first_decision"]),
}

(ddir / "dev_choices.json").write_text(json.dumps(dev_choices, indent=2, default=str))
record, freeze_status = fz.write_or_verify(cfg, paths, dev_choices, MODE)
print(f"Freeze status: {freeze_status} (record created {record['created_utc']})")
print("experiment SHA-256:", record["experiment_sha256"][:16], "… | code SHA-256:", record["code_sha256"][:16], "…")

## 5. Test-period forecasts with the frozen settings

From here on nothing is chosen. Each test month-end decision uses the frozen static duration and half-life, refits the AR and economic models on all data up to that date, and records when its information was available, which training data it used, and when it executes.

In [ ]:
if MODE == "development":
    print("Development mode: stopping before any test-period computation. Rerun in full mode to continue.")
else:
    fc_test = pl.forecasts(monthly, win, "test", static_d, halflife, models_list)
    delay = EXP["hedging"]["base_delay_days"]
    pos = daily.index.searchsorted(pd.to_datetime(fc_test["decision_date"]))
    fc_test["execution_date"] = [daily.index[min(i + delay, len(daily) - 1)] for i in pos]
    fc_test["estimation_period_of_last_realized"] = fc_test["decision_month"].astype(str)
    fc_test.to_csv(fdir / "forecasts_test.csv", index=False)
    cols = ["model", "decision_month", "target_month", "decision_date", "available_at", "training_cutoff_month",
            "execution_date", "n_train", "forecast", "incentive_source", "infeasible"]
    display(fc_test[cols].head(8))
    fz.log_test_evaluation(paths, record, "06", "test-period forecasts computed (no evaluation)")

## 6. Look-ahead check on the real data

For several cut dates in the test period, every input that was not yet available at the cut (prices, yields, and PMMS dated after it, and NMDB quarters *released* after it, even if their quarter ended before it) is replaced with noise and the whole pipeline is rerun. Every forecast and position dated on or before the cut must be identical to machine precision, while later ones should change.

A second check uses a cut inside the development period. There the retrospective proxy's calibration uses NMDB data released after the cut, so some development forecasts are *expected* to change: that is exactly why they are labeled retrospective.

In [ ]:
if MODE != "development":
    tables = load_raw_tables(RawStore(paths.raw), cfg)
    test_dates = daily.loc[str(win["test_start"]):].index
    cuts = [test_dates[int(len(test_dates) * q)] for q in (0.2, 0.5, 0.8)]
    results = [pl.perturbation_check(tables, cfg, EXP, c, seed=EXP["seed"] + k, halflife=halflife) for k, c in enumerate(cuts)]
    dev_cut = pd.Timestamp("2012-06-29")
    dev_check = pl.perturbation_check(tables, cfg, EXP, dev_cut, seed=EXP["seed"] + 99, halflife=halflife, segment="dev")
    look = {"test_period_checks": results, "development_period_check": dev_check}
    (fdir / "lookahead_check.json").write_text(json.dumps(look, indent=2, default=str))
    display(pd.DataFrame(results))
    print("Development-period cut (retrospective proxy expected to leak):", dev_check)
    assert all(r["passed"] for r in results), "Look-ahead check failed in the test period"